# Professor-Feedback Results
**Source:** `results_prof/` (separate from `results_v2/`)

New experiments:
- Yelp α=0.01 — 5 methods × 5 seeds × **40 rounds**
- GSM8K α=0.5 — seeds 45, 46 fill-in (→ 5 seeds total)
- GSM8K α=0.01 — 5 methods × 5 seeds × **40 rounds**

Run cells top-to-bottom. Missing runs show as `—` in tables.

In [ ]:
import json, os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 150, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

# ── paths ──────────────────────────────────────────────────────────────────────
REWORK   = os.path.expanduser('~/FedLLM-Re/rework')
PROF_DIR = os.path.join(REWORK, 'results_prof')
V2_DIR   = os.path.join(REWORK, 'results_v2')   # baseline for comparison

DATASET_PATHS = {
    'yelp':  [os.path.join(PROF_DIR, 'yelp'),  os.path.join(V2_DIR, 'yelp')],
    'gsm8k': [os.path.join(PROF_DIR, 'gsm8k'), os.path.join(V2_DIR, 'gsm8k')],
}

PRIMARY_METRIC = {'yelp': 'accuracy', 'gsm8k': 'exact_match'}
METRIC_LABEL   = {'yelp': 'Accuracy (%)', 'gsm8k': 'Exact Match (%)'}

METHODS = ['homo_r8', 'hetero_pad', 'flexlora', 'hetlora', 'hetlora_m']
LABELS  = {
    'homo_r8':    'Homo r=8',
    'hetero_pad': 'Hetero-Pad',
    'flexlora':   'FlexLoRA',
    'hetlora':    'HetLoRA',
    'hetlora_m':  'HetLoRA-M (Ours)',
}
COLORS = {
    'homo_r8':    '#9e9e9e',
    'hetero_pad': '#ff9800',
    'flexlora':   '#2196f3',
    'hetlora':    '#9c27b0',
    'hetlora_m':  '#e53935',
}
LINESTYLES  = {'homo_r8': '--', 'hetero_pad': '-.', 'flexlora': ':', 'hetlora': '-', 'hetlora_m': '-'}
LINEWIDTHS  = {'homo_r8': 1.5, 'hetero_pad': 1.5, 'flexlora': 1.5, 'hetlora': 1.5, 'hetlora_m': 2.5}

print('Paths:')
for ds, paths in DATASET_PATHS.items():
    for p in paths:
        count = len(glob.glob(os.path.join(p, '*.json'))) if os.path.exists(p) else 0
        print(f'  {p}  [{count} files]')

In [ ]:
# ── loaders ────────────────────────────────────────────────────────────────────

def load_dataset(dataset):
    metric = PRIMARY_METRIC[dataset]
    rows, seen = [], set()
    for path in DATASET_PATHS[dataset]:
        if not os.path.exists(path):
            continue
        for fp in sorted(glob.glob(os.path.join(path, '*.json'))):
            try:
                data = json.loads(open(fp).read())
            except Exception as e:
                print(f'  WARNING {fp}: {e}')
                continue
            method = data.get('method', 'unknown')
            seed   = data.get('seed', -1)
            alpha  = data.get('alpha', -1)
            key    = (method, alpha, seed)
            if key in seen:
                continue
            seen.add(key)
            for r in data.get('rounds', []):
                val = r.get(metric)
                if val is None:
                    val = r.get('accuracy')
                if val is None:
                    continue
                rows.append({
                    'dataset': dataset, 'method': method,
                    'alpha': alpha, 'seed': seed,
                    'round': r.get('round', r.get('communication_round', 0)),
                    'value': float(val),
                    'source': 'prof' if 'results_prof' in fp else 'v2',
                })
    return pd.DataFrame(rows)


def summary_stats(df):
    if df.empty:
        return pd.DataFrame()
    rows = []
    for (method, alpha, seed), g in df.groupby(['method', 'alpha', 'seed']):
        vals = g.sort_values('round')['value'].values
        n    = len(vals)
        auc  = np.mean(vals)
        ml5  = np.mean(vals[-5:]) if n >= 5 else np.mean(vals)
        best = np.max(vals)
        rows.append({'method': method, 'alpha': alpha, 'seed': seed,
                     'auc': auc, 'mean_l5': ml5, 'best': best, 'n_rounds': n})
    per_seed = pd.DataFrame(rows)
    # aggregate across seeds
    agg = []
    for (method, alpha), g in per_seed.groupby(['method', 'alpha']):
        agg.append({
            'method': method, 'alpha': alpha, 'n_seeds': len(g),
            'auc':      g['auc'].mean(),      'auc_std':      g['auc'].std(ddof=1) if len(g)>1 else 0,
            'mean_l5':  g['mean_l5'].mean(),  'mean_l5_std':  g['mean_l5'].std(ddof=1) if len(g)>1 else 0,
            'best':     g['best'].mean(),      'best_std':     g['best'].std(ddof=1) if len(g)>1 else 0,
            'n_rounds': g['n_rounds'].mean(),
        })
    return pd.DataFrame(agg)


def mean_curve(df, method, alpha):
    sub = df[(df['method'] == method) & (df['alpha'] == alpha)]
    if sub.empty:
        return None, None, None
    pivot = sub.pivot_table(index='round', columns='seed', values='value')
    mu  = pivot.mean(axis=1).values
    std = pivot.std(axis=1).values
    return pivot.index.values, mu * 100, std * 100


# load
dfs   = {ds: load_dataset(ds) for ds in ['yelp', 'gsm8k']}
stats = {ds: summary_stats(dfs[ds]) for ds in dfs}

for ds, df in dfs.items():
    if df.empty:
        print(f'{ds}: NO DATA')
    else:
        alphas   = sorted(df['alpha'].unique())
        methods  = df['method'].unique().tolist()
        seeds    = sorted(df['seed'].unique())
        print(f'{ds}: {len(df)} rows | alphas={alphas} | seeds={seeds}')
        print(f'      methods={methods}')

In [ ]:
# ── PROGRESS TRACKER ───────────────────────────────────────────────────────────
# Shows which (method × alpha × seed) runs are done vs still running.

EXPECTED = {
    'yelp':  {'alphas': [0.01, 0.1, 0.5], 'seeds': [42,43,44,45,46]},
    'gsm8k': {'alphas': [0.01, 0.5],       'seeds': [42,43,44,45,46]},
}

print(f'{'Dataset':<8} {'Method':<14} {'Alpha':>6}  Seeds done / expected')
print('-' * 70)
total_done = total_exp = 0

for ds in ['yelp', 'gsm8k']:
    df = dfs[ds]
    for m in METHODS:
        for alpha in EXPECTED[ds]['alphas']:
            exp_seeds = EXPECTED[ds]['seeds']
            done_seeds = []
            missing_seeds = []
            for s in exp_seeds:
                has = not df[(df['method']==m)&(df['alpha']==alpha)&(df['seed']==s)].empty
                (done_seeds if has else missing_seeds).append(s)
            total_done += len(done_seeds)
            total_exp  += len(exp_seeds)
            bar = '█' * len(done_seeds) + '░' * len(missing_seeds)
            miss_str = f'  missing: {missing_seeds}' if missing_seeds else '  ✓ complete'
            print(f'{ds:<8} {LABELS.get(m,m):<14} α={alpha:>5}  [{bar}] {len(done_seeds)}/{len(exp_seeds)}{miss_str}')

print()
print(f'Overall: {total_done}/{total_exp} runs complete ({100*total_done/total_exp:.0f}%)')

---
## 1  Yelp — α=0.01 (New Hardest Setting)

In [ ]:
# ── Yelp convergence curves: α=0.01 vs α=0.1 side-by-side ────────────────────
ds  = 'yelp'
df  = dfs[ds]
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

for ax, alpha in zip(axes, [0.5, 0.1, 0.01]):
    for m in METHODS:
        rounds, mu, std = mean_curve(df, m, alpha)
        if rounds is None:
            continue
        ax.plot(rounds, mu, label=LABELS[m], color=COLORS[m],
                ls=LINESTYLES[m], lw=LINEWIDTHS[m])
        ax.fill_between(rounds, mu-std, mu+std, alpha=0.12, color=COLORS[m])
    ax.set_title(f'Yelp  α={alpha}', fontweight='bold')
    ax.set_xlabel('Round')
    ax.set_ylabel(METRIC_LABEL[ds])
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(REWORK, 'figures/yelp_prof_convergence.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Yelp summary table ─────────────────────────────────────────────────────────
st = stats['yelp']

def fmt(mean, std, pct=True, n=None):
    s = 100 if pct else 1
    dp = 1 if pct else 3
    if n is not None and n < 2:
        return f'{mean*s:.{dp}f} (n={n})'
    return f'{mean*s:.{dp}f}±{std*s:.{dp}f}'

rows = []
for m in METHODS:
    row = {'Method': LABELS.get(m, m)}
    for alpha in [0.5, 0.1, 0.01]:
        sub = st[(st['method']==m) & (st['alpha']==alpha)]
        if sub.empty:
            row[f'AUC α={alpha}']    = '—'
            row[f'MeanL5 α={alpha}'] = '—'
        else:
            r = sub.iloc[0]
            n = int(r['n_seeds'])
            row[f'AUC α={alpha}']    = fmt(r['auc'],     r['auc_std'],     n=n)
            row[f'MeanL5 α={alpha}'] = fmt(r['mean_l5'], r['mean_l5_std'], n=n)
    rows.append(row)

result_df = pd.DataFrame(rows).set_index('Method')

# bold the best AUC per alpha column
print('Yelp Results — AUC / Mean-L5 (%, ★ AUC = primary metric)')
display(result_df)

In [ ]:
# ── Yelp: heterogeneity sweep bar chart (AUC vs alpha) ────────────────────────
# Shows how each method degrades as α decreases (0.5 → 0.1 → 0.01).
# HetLoRA-M should degrade least.

alphas = [0.5, 0.1, 0.01]
x = np.arange(len(alphas))
width = 0.15
offsets = np.linspace(-(len(METHODS)-1)/2, (len(METHODS)-1)/2, len(METHODS)) * width

fig, ax = plt.subplots(figsize=(9, 5))
st = stats['yelp']

for i, m in enumerate(METHODS):
    vals, errs = [], []
    for alpha in alphas:
        sub = st[(st['method']==m) & (st['alpha']==alpha)]
        if sub.empty:
            vals.append(0); errs.append(0)
        else:
            r = sub.iloc[0]
            vals.append(r['auc'] * 100)
            errs.append(r['auc_std'] * 100)
    ax.bar(x + offsets[i], vals, width, yerr=errs, capsize=3,
           label=LABELS[m], color=COLORS[m], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([f'α={a}' for a in alphas])
ax.set_ylabel('AUC (%, mean over rounds)')
ax.set_title('Yelp: AUC across heterogeneity levels')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(REWORK, 'figures/yelp_prof_alpha_sweep.pdf'), bbox_inches='tight')
plt.show()

---
## 2  GSM8K — α=0.01 + 5-seed fill

In [ ]:
# ── GSM8K convergence curves ───────────────────────────────────────────────────
ds  = 'gsm8k'
df  = dfs[ds]
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)

for ax, alpha in zip(axes, [0.5, 0.01]):
    for m in METHODS:
        rounds, mu, std = mean_curve(df, m, alpha)
        if rounds is None:
            continue
        ax.plot(rounds, mu, label=LABELS[m], color=COLORS[m],
                ls=LINESTYLES[m], lw=LINEWIDTHS[m])
        ax.fill_between(rounds, mu-std, mu+std, alpha=0.12, color=COLORS[m])
    ax.set_title(f'GSM8K  α={alpha}', fontweight='bold')
    ax.set_xlabel('Round')
    ax.set_ylabel(METRIC_LABEL[ds])
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(REWORK, 'figures/gsm8k_prof_convergence.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# ── GSM8K summary table ────────────────────────────────────────────────────────
st = stats['gsm8k']
rows = []
for m in METHODS:
    row = {'Method': LABELS.get(m, m)}
    for alpha in [0.5, 0.01]:
        sub = st[(st['method']==m) & (st['alpha']==alpha)]
        if sub.empty:
            row[f'AUC α={alpha}']    = '—'
            row[f'MeanL5 α={alpha}'] = '—'
            row[f'n seeds α={alpha}'] = 0
        else:
            r = sub.iloc[0]
            n = int(r['n_seeds'])
            row[f'AUC α={alpha}']    = fmt(r['auc'],     r['auc_std'],     n=n)
            row[f'MeanL5 α={alpha}'] = fmt(r['mean_l5'], r['mean_l5_std'], n=n)
            row[f'n seeds α={alpha}'] = n
    rows.append(row)

print('GSM8K Results — AUC / Mean-L5 (%)')
display(pd.DataFrame(rows).set_index('Method'))

---
## 3  Mean-Last-5 Std — Convergence Evidence

In [ ]:
# ── Per-round std over last 5 rounds (convergence stability table) ─────────────
# For each (method × alpha), compute std of accuracy across rounds in the last 5.
# Low std = converged. High std = still oscillating.

print('Last-5-rounds std (pp) — stability at convergence')
print('Low = converged, High = still oscillating\n')

for ds in ['yelp', 'gsm8k']:
    df = dfs[ds]
    if df.empty:
        continue
    print(f'=== {ds.upper()} ===')
    rows = []
    for m in METHODS:
        row = {'Method': LABELS.get(m, m)}
        alphas_avail = sorted(df[df['method']==m]['alpha'].unique())
        for alpha in alphas_avail:
            sub = df[(df['method']==m) & (df['alpha']==alpha)]
            # last-5 std averaged across seeds
            seed_stds = []
            for seed, g in sub.groupby('seed'):
                last5 = g.sort_values('round')['value'].values[-5:]
                if len(last5) >= 2:
                    seed_stds.append(np.std(last5, ddof=1) * 100)
            if seed_stds:
                row[f'α={alpha}'] = f'{np.mean(seed_stds):.2f}pp'
            else:
                row[f'α={alpha}'] = '—'
        rows.append(row)
    display(pd.DataFrame(rows).set_index('Method'))
    print()

---
## 4  LaTeX — Updated Table for Paper

In [ ]:
# ── LaTeX snippet: Yelp + GSM8K × {α=0.5, α=0.1, α=0.01} ────────────────────
# Copy into submission_v4.tex

def fmt_tex(st, m, alpha, pct=True):
    sub = st[(st['method']==m) & (st['alpha']==alpha)]
    if sub.empty:
        return '—'
    r  = sub.iloc[0]
    n  = int(r['n_seeds'])
    s  = 100 if pct else 1
    dp = 1
    mu, sd = r['auc']*s, r['auc_std']*s
    if n < 2:
        return f'{mu:.{dp}f}'
    return f'{mu:.{dp}f}$\\pm${sd:.{dp}f}'

def best_method(st, alpha):
    best_m, best_v = None, -1
    for m in METHODS:
        sub = st[(st['method']==m) & (st['alpha']==alpha)]
        if sub.empty: continue
        v = sub.iloc[0]['auc']
        if v > best_v:
            best_v, best_m = v, m
    return best_m

lines = []
lines.append(r'\begin{table}[t]')
lines.append(r'\centering')
lines.append(r'\caption{Main results (AUC = mean accuracy over all rounds, \%). ' +
             r'\textbf{Bold} = best per column. $\dagger$ = 5 seeds; others 3 seeds.}')
lines.append(r'\label{tab:main_results}')
lines.append(r'\small')
lines.append(r'\begin{tabular}{lcccccc}')
lines.append(r'\toprule')
lines.append(r'& \multicolumn{3}{c}{\textbf{Yelp (Acc \%)}} & \multicolumn{2}{c}{\textbf{GSM8K (EM \%)}} \\')
lines.append(r'\cmidrule(lr){2-4} \cmidrule(lr){5-6}')
lines.append(r'\textbf{Method} & $\alpha$=0.5 & $\alpha$=0.1 & $\alpha$=0.01 & $\alpha$=0.5 & $\alpha$=0.01 \\')
lines.append(r'\midrule')

yelp_st  = stats['yelp']
gsm8k_st = stats['gsm8k']
bests = {
    ('yelp',  0.5):  best_method(yelp_st,  0.5),
    ('yelp',  0.1):  best_method(yelp_st,  0.1),
    ('yelp',  0.01): best_method(yelp_st,  0.01),
    ('gsm8k', 0.5):  best_method(gsm8k_st, 0.5),
    ('gsm8k', 0.01): best_method(gsm8k_st, 0.01),
}

for m in METHODS:
    cells = []
    for ds, alpha, pct in [
        ('yelp',  0.5,  True), ('yelp',  0.1,  True), ('yelp',  0.01, True),
        ('gsm8k', 0.5,  True), ('gsm8k', 0.01, True),
    ]:
        st_ = yelp_st if ds == 'yelp' else gsm8k_st
        cell = fmt_tex(st_, m, alpha, pct)
        if bests.get((ds, alpha)) == m and cell != '—':
            cell = f'\\textbf{{{cell}}}'
        cells.append(cell)
    label = LABELS[m].replace('(Ours)', '(Ours)$\\dagger$') if m == 'hetlora_m' else LABELS[m]
    lines.append(f'{label} & {" & ".join(cells)} \\\\')

lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
lines.append(r'\end{table}')

print('\n'.join(lines))

In [ ]:
# ── Quick sanity: how many rounds did each run actually complete? ───────────────
# Should be 40 for alpha=0.01, 20 for alpha=0.5/0.1
for ds in ['yelp', 'gsm8k']:
    df = dfs[ds]
    if df.empty:
        continue
    print(f'=== {ds.upper()} — rounds per run ===')
    summary = (df.groupby(['method', 'alpha', 'seed'])['round']
               .max().reset_index()
               .rename(columns={'round': 'max_round'}))
    pivot = summary.pivot_table(index=['method','alpha'], columns='seed',
                                values='max_round', aggfunc='first')
    print(pivot.to_string())
    print()